# Atlantic Forest restoration costs: bootstrap Monte Carlo analysis

This notebook reproduces the workflow used to assess uncertainty in estimated mean input prices for a hypothetical 1-ha active restoration project in Vitória da Conquista, Bahia, Brazil. **The project budget is read directly from the Excel workbook**, and each of eight selected inputs has 10 observed price quotations.

The Excel workbook contains the detailed establishment and four-year maintenance costs. Values are displayed to **two decimal places**, while calculations retain their original numeric precision. Run all cells in sequence from the repository directory or its `notebooks` subdirectory.

## 1. Load libraries and project data

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

current=Path.cwd().resolve()
ROOT=next((p for p in [current,current.parent] if (p/'src'/'restoration_monte_carlo.py').exists()),None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from the repository or notebooks/ directory.')
sys.path.insert(0,str(ROOT/'src'))
import restoration_monte_carlo as model

budget=model.read_budget(ROOT/'data'/'restoration_project_budget.xlsx')
quotes=model.read_quotations(ROOT/'data'/'input_price_quotations.csv')
summary_budget=pd.DataFrame({
    'Component':['Inputs','Services','Total project'],
    'Cost (BRL)':[budget['input_cost'],budget['service_cost'],budget['total_cost']]
})
display(summary_budget.style.format({'Cost (BRL)':'{:,.2f}'}))
print(f"Brush-cutter: {budget['brush_hours'].sum():.2f} service hours × BRL {budget['brush_rate']:.2f}/h = BRL {budget['brush_cost']:,.2f}")
assert np.isclose(budget['total_cost'],28455.13,atol=0.02)
assert len(quotes)==80

## 2. Bootstrap Monte Carlo simulation

In each of 10,000 iterations, ten quotations are independently resampled **with replacement** for each of the eight stochastic inputs. Their bootstrap mean prices are multiplied by the budgeted quantities read from the spreadsheet. Seedlings and service costs remain fixed. The random seed allows this run to be replicated; it does not establish that the same sequence of random draws was used to generate the published figures.

In [ ]:
SEED=123
ITERATIONS=10_000
sim=model.simulate(seed=SEED,n=ITERATIONS,budget=budget,quotes=quotes)
display(sim[['Iteration','Input costs','Service costs','Total cost']].head().style.format({'Input costs':'{:,.2f}','Service costs':'{:,.2f}','Total cost':'{:,.2f}'}))
assert len(sim)==ITERATIONS
assert np.allclose(sim['Total cost'],sim['Input costs']+sim['Service costs'])

## 3. Simulated cost distribution and risk metrics

The resulting cost distribution summarizes uncertainty arising from the estimated mean prices of surveyed inputs. The outputs include the mean, standard deviation, coefficient of variation, percentiles and CVaR95 (the mean cost at or above the 95th percentile). These estimates do not represent all possible future operational risks.

In [ ]:
indicators=model.metrics(sim['Total cost'])
display(pd.DataFrame({'Metric':list(indicators),'Value':list(indicators.values())}).style.format({'Value':'{:,.2f}'}))
fig,ax=plt.subplots(figsize=(9,4.8))
ax.hist(sim['Total cost'],bins=45,edgecolor='white')
ax.axvline(indicators['Mean'],linestyle='--',label='Mean')
ax.axvline(indicators['P95 (VaR95)'],linestyle=':',label='95th percentile')
ax.set(xlabel='Restoration cost (BRL/ha)',ylabel='Frequency',title='Simulated restoration cost distribution')
ax.legend();plt.tight_layout();plt.show()

## 4. Convergence of simulation indicators

In [ ]:
conv=model.convergence(sim)
display(conv.style.format({'Mean':'{:,.2f}','P95':'{:,.2f}','CV (%)':'{:,.2f}'}))
fig,ax=plt.subplots(figsize=(9,4.5))
ax.plot(conv['Iterations'],conv['Mean'],marker='o',label='Mean')
ax.plot(conv['Iterations'],conv['P95'],marker='s',label='P95')
ax.set(xlabel='Number of Monte Carlo iterations',ylabel='Cost (BRL/ha)',title='Convergence of cost indicators')
ax.legend();plt.tight_layout();plt.show()

## 5. Price sensitivity

Pearson correlations are calculated between the bootstrapped mean price of each input and the simulated total project cost.

In [ ]:
sens=model.sensitivity(sim,budget)
display(sens.style.format({'Pearson r':'{:.2f}','Budgeted quantity':'{:,.2f}'}))
fig,ax=plt.subplots(figsize=(9,4.8))
order=sens.iloc[::-1]
ax.barh(order['Input'],order['Pearson r'])
ax.set(xlabel='Pearson correlation',title='Sensitivity of total cost to sampled mean input prices')
plt.tight_layout();plt.show()